In [1]:
!pip install --upgrade --force-reinstall -q mujoco mediapy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.18.1 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 17.4 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 17.4 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.107 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompati

In [2]:
import os
# Configure MuJoCo to use the Colab GPU for background rendering
os.environ["MUJOCO_GL"] = "egl"


import mujoco
import mediapy as media
import numpy as np

In [3]:
# 1. Define the 7-DOF Robot Arm structure
arm_7dof_xml = """
<mujoco>
    <option timestep="0.002" gravity="0 0 -9.81"/>
    <worldbody>
        <light pos="0 0 4" dir="0 0 -1" castshadow="true"/>
        <geom type="plane" size="2 2 .1" rgba=".9 .9 .9 1"/>

        <body name="base" pos="0 0 0">
            <geom type="cylinder" size="0.12 0.1" rgba="0.2 0.2 0.2 1" pos="0 0 0.05"/>

            <!-- SHOULDER SECTION (3 DOF) -->
            <body name="link1" pos="0 0 0.1">
                <joint name="joint1_shoulder_yaw" type="hinge" axis="0 0 1"/>
                <geom type="cylinder" size="0.08 0.1" rgba="0.3 0.3 0.8 1" pos="0 0 0.1"/>

                <body name="link2" pos="0 0 0.2">
                    <joint name="joint2_shoulder_pitch" type="hinge" axis="0 1 0"/>
                    <geom type="capsule" fromto="0 0 0 0 0 0.3" size="0.07" rgba="0.3 0.3 0.8 1"/>

                    <body name="link3" pos="0 0 0.3">
                        <joint name="joint3_shoulder_roll" type="hinge" axis="0 0 1"/>
                        <geom type="cylinder" size="0.06 0.08" rgba="0.3 0.3 0.8 1" pos="0 0 0.08"/>

                        <!-- ELBOW SECTION (1 DOF) -->
                        <body name="link4" pos="0 0 0.16">
                            <joint name="joint4_elbow_pitch" type="hinge" axis="0 1 0"/>
                            <geom type="capsule" fromto="0 0 0 0 0 0.3" size="0.05" rgba="0.8 0.3 0.3 1"/>

                            <!-- WRIST SECTION (3 DOF) -->
                            <body name="link5" pos="0 0 0.3">
                                <joint name="joint5_wrist_roll" type="hinge" axis="0 0 1"/>
                                <geom type="cylinder" size="0.04 0.06" rgba="0.3 0.8 0.3 1" pos="0 0 0.06"/>

                                <body name="link6" pos="0 0 0.12">
                                    <joint name="joint6_wrist_pitch" type="hinge" axis="0 1 0"/>
                                    <geom type="capsule" fromto="0 0 0 0 0 0.15" size="0.035" rgba="0.3 0.8 0.3 1"/>

                                    <body name="link7" pos="0 0 0.15">
                                        <joint name="joint7_wrist_yaw" type="hinge" axis="1 0 0"/>
                                        <geom type="sphere" size="0.04" rgba="0.9 0.9 0.1 1"/>
                                    </body>
                                </body>
                            </body>
                        </body>
                    </body>
                </body>
            </body>
        </body>
    </worldbody>
</mujoco>
"""


# 2. Compile model and allocate data
model = mujoco.MjModel.from_xml_string(arm_7dof_xml)
data = mujoco.MjData(model)

In [4]:
# 3. Setup video render properties
renderer = mujoco.Renderer(model, height=480, width=640)
frames = []
duration = 10.0
fps = 30
frame_interval = int(1.0 / (fps * model.opt.timestep))


# 4. Simulation Loop
print("Simulating 7-DOF movement...")
while data.time < duration:
    # Send custom dynamic sinusoidal inputs to all 7 joints
    # data.qvel = [
    #     0.5 * np.sin(1.0 * np.pi * data.time),
    #     0.4 * np.cos(1.5 * np.pi * data.time),
    #     0.6 * np.sin(0.8 * np.pi * data.time),
    #     0.8 * np.sin(2.0 * np.pi * data.time),
    #     0.5 * np.cos(1.2 * np.pi * data.time),
    #     0.7 * np.sin(2.5 * np.pi * data.time),
    #     1.0 * np.cos(3.0 * np.pi * data.time)
    # ]

    data.qvel = [
        0.05 * np.sin(1.0 * np.pi * data.time),
        1 + 0.06 * np.cos(1.5 * np.pi * data.time),
        0.06 * np.sin(0.8 * np.pi * data.time),
        -0.5 + 0.05 * np.sin(2.0 * np.pi * data.time),
        0.5 * np.cos(1.2 * np.pi * data.time),
        -0.4 + 0.4 * np.sin(2.5 * np.pi * data.time),
        0.02 * np.cos(3.0 * np.pi * data.time)
    ]

    mujoco.mj_step(model, data)

    if int(data.time / model.opt.timestep) % frame_interval == 0:
        renderer.update_scene(data, camera=-1)
        frames.append(renderer.render())




Simulating 7-DOF movement...


In [5]:
print("Displaying animation:")
if frames:
    media.show_video(frames, fps=fps)
else:
    print("No frames were generated. Please check the simulation loop in the previous cell.")

Displaying animation:


This browser does not support the video tag.


In [6]:
# ============================================================
# PAD-driven robot arm animation
# Pleasure  (P): valence      -> up/open (P>0) vs droop/closed (P<0)
# Arousal   (A): energy level -> speed/frequency of motion
# Dominance (D): expansiveness-> amplitude/size of motion
# ============================================================

def pad_qvel(t, duration, P, A, D):
    """
    Compute the 7-joint velocity vector at time t, shaped by PAD (each in [-1,1]).
    """
    # Arousal -> how fast the joints oscillate (low A = sluggish, high A = energetic)
    freq_scale = 0.5 + 1.5 * (A + 1) / 2      # A=-1 -> 0.5x speed, A=1 -> 2.0x speed

    # Dominance -> how large/expansive the motion is
    amp_scale = 0.3 + 1.7 * (D + 1) / 2       # D=-1 -> 0.3x size, D=1 -> 2.0x size

    # Pleasure -> directional bias: droop (negative) vs lift/open (positive)
    bias_scale = 0.08 * P

    # Envelope: bias smoothly rises then returns to 0 over the full duration,
    # so joints don't spin away indefinitely -- they settle back near neutral.
    recover = np.sin(np.pi * t / duration)

    base_freqs = [1.0, 1.5, 0.8, 2.0, 1.2, 2.5, 3.0]
    base_amps  = [0.05, 0.06, 0.06, 0.05, 0.05, 0.05, 0.02]
    trig_fns   = [np.sin, np.cos, np.sin, np.sin, np.cos, np.sin, np.cos]
    has_bias   = [False, True, False, True, False, True, False]  # pitch joints

    qvel = []
    for i in range(7):
        osc = amp_scale * base_amps[i] * trig_fns[i](base_freqs[i] * freq_scale * np.pi * t)
        bias = bias_scale * recover if has_bias[i] else 0.0
        qvel.append(bias + osc)
    return qvel


def simulate_pad(model, P, A, D, duration=8.0, fps=30, label=""):
    """
    Runs one full simulation for a given PAD triplet and returns the frame list.
    """
    data = mujoco.MjData(model)   # fresh state so conditions don't leak into each other
    renderer = mujoco.Renderer(model, height=480, width=640)
    frames = []
    frame_interval = int(1.0 / (fps * model.opt.timestep))

    print(f"Simulating {label}: PAD=({P},{A},{D})")
    while data.time < duration:
        data.qvel = pad_qvel(data.time, duration, P, A, D)
        mujoco.mj_step(model, data)

        if int(data.time / model.opt.timestep) % frame_interval == 0:
            renderer.update_scene(data, camera=-1)
            frames.append(renderer.render())

    renderer.close()
    return frames


# ------------------------------------------------------------
# Run all required PAD conditions and display each movie
# ------------------------------------------------------------
pad_conditions = [
    ("Baseline",        (0, 0, 0)),
    ("PAD = [1,1,1]",   (1, 1, 1)),
    ("PAD = [-1,-1,-1]",(-1, -1, -1)),
    ("PAD = [-1,1,-1]", (-1, 1, -1)),
    ("PAD = [-1,1,1]",  (-1, 1, 1)),
]

duration = 8.0
fps = 30

for label, (P, A, D) in pad_conditions:
    frames = simulate_pad(model, P, A, D, duration=duration, fps=fps, label=label)
    print(f"Displaying {label}:")
    media.show_video(frames, fps=fps)

Simulating Baseline: PAD=(0,0,0)
Displaying Baseline:


This browser does not support the video tag.


Simulating PAD = [1,1,1]: PAD=(1,1,1)
Displaying PAD = [1,1,1]:


This browser does not support the video tag.


Simulating PAD = [-1,-1,-1]: PAD=(-1,-1,-1)
Displaying PAD = [-1,-1,-1]:


This browser does not support the video tag.


Simulating PAD = [-1,1,-1]: PAD=(-1,1,-1)
Displaying PAD = [-1,1,-1]:


This browser does not support the video tag.


Simulating PAD = [-1,1,1]: PAD=(-1,1,1)
Displaying PAD = [-1,1,1]:


This browser does not support the video tag.
